In [1]:
using Sunny, GLMakie, ColorSchemes, Colors, LinearAlgebra, ProgressMeter, Distributed

include("Frida_code/func_clino1.jl")
units = Units(:meV, :angstrom)


Units(:meV, :angstrom)

In [9]:
ncores = length(Sys.cpu_info())
addprocs(ncores-1)

7-element Vector{Int64}:
 2
 3
 4
 5
 6
 7
 8

In [ ]:
# cif = joinpath(@__DIR__, "Ho2Ti2O7_mp-33948_symmetrized.cif")
# crystal = Crystal(cif; symprec=0.001)
# Ho_sub = subcrystal(crystal, "Ho0")

# dims = (3, 3, 3)  # Supercell dimensions
# A = 18.7 / 3
# B = A
# momentinfo = [1 => Moment(s=8, g=[A B B
#                                     B A B
#                                     B B A])]  # Specify local moment information, note that all sites are symmetry equivalent
# # momentinfo = [1 => Moment(s=8, g=2)]  
# sys_Ho = System(Ho_sub, momentinfo, :dipole; dims); # Same on MgCr2O4 crystal

# J = 0.52/ 11.604 # (meV)
# set_exchange!(sys_Ho, J, Bond(1, 3, [0, 0, 0]))

# # S = spin_matrices(spin_label(sys_Ho, 1))
# # O = stevens_matrices(spin_label(sys_Ho, 1))
# # D = 1
# # set_onsite_coupling!(sys_Ho, -D *(O[2,-2]+2*O[2,-1]+2*O[2,1]), 1) 

# muB = 10.6
# vac_per = 0.6745817653324668
# enable_dipole_dipole!(sys_Ho, units.vacuum_permeability*muB^2)

# randomize_spins!(sys_Ho)
# minimize_energy!(sys_Ho)
# plot_spins(sys_Ho; color=[S[3] for S in sys_Ho.dipoles])

In [10]:
@everywhere using Sunny, LinearAlgebra, ProgressMeter

@everywhere function make_system(seed)
    cif = joinpath(@__DIR__, "Ho2Ti2O7_mp-33948_symmetrized.cif")
    crystal = Crystal(cif; symprec=0.001)
    Ho_sub = subcrystal(crystal, "Ho0")

    dims = (3, 3, 3)  # Supercell dimensions
    A = 18.7 / 3
    B = A
    momentinfo = [1 => Moment(s=8, g=[A B B
                                      B A B
                                      B B A])]  # Specify local moment information, note that all sites are symmetry equivalent
    # momentinfo = [1 => Moment(s=8, g=2)]  
    sys_Ho = System(Ho_sub, momentinfo, :dipole; dims); # Same on MgCr2O4 crystal

    J = 0.52/ 11.604 # (meV)
    set_exchange!(sys_Ho, J, Bond(1, 3, [0, 0, 0]))

    # S = spin_matrices(spin_label(sys_Ho, 1))
    # O = stevens_matrices(spin_label(sys_Ho, 1))
    # D = 1
    # set_onsite_coupling!(sys_Ho, -D *(O[2,-2]+2*O[2,-1]+2*O[2,1]), 1) 

    muB = 10.6
    vac_per = 0.6745817653324668
    enable_dipole_dipole!(sys_Ho, vac_per*muB^2)

    return sys_Ho
end

sys_Ho = make_system(0)

System [Dipole mode]
Supercell (3×3×3)×16
Energy per site -1319.


In [5]:
# view_crystal(sys_Ho)

## Langevin

In [6]:
kT      = 1.7 / 11            
tol     = 1e-3                 
nsteps  = 1000

dt      = 2.811e-05
damping = 0.1     
kT      = 1.7/11


sys_i = reshape_supercell(sys_Ho, [10 0 0; 0 10 0; 0 0 10]) 
randomize_spins!(sys_i)

fig2 = Figure(size=(1000, 500))

langevin = Langevin(dt; damping, kT)
suggest_timestep(sys_i, langevin; tol=tol)

energies = [energy_per_site(sys_i)]
for _ in 1:nsteps
    step!(sys_i, langevin)
    push!(energies, energy_per_site(sys_i))
end
ax = Axis(fig2[1,1];
    xlabel = "Timesteps",
    ylabel = "Energy (meV)",
    title  = "dt = $(dt), damping = $(damping), tol = $(tol)")
lines!(ax, energies, color=:red)


fig2


InterruptException: InterruptException:

In [ ]:
tol     = 1e-3                 
nsteps  = 1000

kT      = 1.7*Sunny.meV_per_K 
damping = 0.1

si= 3 # q-space pixel size as 1/si

# Energy range: max resolvable frequency ~ π/dt, but stay physically reasonable

energies = range(0, 5, 100)

scs = @distributed (vcat) for id in 1:ncores
    sys_i = make_system(id)

    randomize_spins!(sys_i)
    minimize_energy!(sys_i)

    sys_i = reshape_supercell(sys_Ho, [si 0 0; 0 si 0; 0 0 si]) 
    randomize_spins!(sys_i)

    formfactors = [1 => FormFactor("Ho3")]
    sc = SampledCorrelations(sys_i; dt, energies, measure=ssf_perp(sys_i; formfactors))

    n_decorr  = nsteps   # steps between samples (matching your thermalization)
    n_samples = 50%ncores       # more samples = smoother spectrum

    langevin = Langevin(dt; damping, kT)

    @showprogress for sample in 1:n_samples #for sample in 1:n_samples
            for _ in 1:n_decorr
                step!(sys_i, langevin)
            end
            add_sample!(sc, sys_i)
    end
    [sc]
end


In [ ]:
sc = merge_correlations(scs)

## Metropolis

In [ ]:
# kT      = 1.7 / 11            
# tol     = 1e-3                 
# nsteps  = 100

# dt      = 0.001521
# damping = 0.1     
# kT      = 1.7/11


# sys_i = reshape_supercell(sys_Ho, [10 0 0; 0 10 0; 0 0 10]) 
# randomize_spins!(sys_i)

# fig2 = Figure(size=(1000, 500))

# metropolis = LocalSampler(kT = kT, nsweeps = 10, propose = propose_flip)
# #suggest_timestep(sys_i, metropolis; tol=tol)

# energies = [energy_per_site(sys_i)]
# for _ in 1:nsteps
#     step!(sys_i, metropolis)
#     push!(energies, energy_per_site(sys_i))
# end
# ax = Axis(fig2[1,1];
#     xlabel = "Timesteps",
#     ylabel = "Energy (meV)",
#     title  = "dt = $(dt), damping = $(damping), tol = $(tol)")
# lines!(ax, energies, color=:red)


# fig2


## Plot

In [ ]:
#Plot (Q,E)-colormap (energies are chosen, q's aren't yet)
fig = Figure(size=(800, 600))
qbs = 0.0


qbs  = 0.0:0.5:1.5
dirs = ["h", "h", "h", "h"]  # adjust per slice
npts = 200

fig = Figure(size=(1200, 900))

for (i, qb) in enumerate(qbs)
    path = [[-4, qb, 0], [4, qb, 0]]
    qpts = q_space_path(crystal, path, npts)
    Sqw  = intensities(sc, qpts; energies=:available, kT)

    xticks, xlabels = qticks_lang(5, path, npts, dirs[i])
    emax = energies[end]

    plot_intensities!(fig[fldmod1(i, 2)...], Sqw;
        title = "K = $qb",
        colormap = :viridis,
        axis = (
            yticks      = 0:emax/emax:emax,
            xticks      = (xticks, xlabels),
            xlabel      = "(H $(qb) 0) (r.l.u.)",
            ylabel      = "Energy (meV)",
        )
    )
end
#Ising-SIA = $(D), 
Label(fig[0, 1:2], "Jnn = $(J), with dipole-dipole \nSystem size: $(si)x$(si)x$(si), tol: $(tol), damping: $(damping), dt: $(dt), T = $(kT/Sunny.meV_per_K) K"; fontsize=15, font=:bold)

fig

UndefVarError: UndefVarError: `crystal` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.
Hint: a global variable of this name also exists in Bravais.

In [ ]:
#Plot integrated over all E (Q,Q)-cut (Static intensities)
qpts = q_space_grid(crystal, [1, 1, 0], range(-6, 6, 200), [0, 0, 1], (-6, 6))
Sq  = intensities_static(sc, qpts; kT)

fig = Figure(; size=(700,500))
#Ising-SIA = $(D),
plot_intensities!(fig[1,1], Sq; title="Jnn = $(J), with dipole-dipole \nStatic/integrated 0<E<$(energies[end]) meV \nSystem size: $(si)x$(si)x$(si), tol: $(tol), damping: $(damping), dt: $(dt), T = $(round(kT/Sunny.meV_per_K; digits=3)) K", colormap=:viridis)
ax = current_axis()
ax.aspect = DataAspect()
fig

UndefVarError: UndefVarError: `crystal` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.
Hint: a global variable of this name also exists in Bravais.